# Shadow-1-Whisper-0.8B — Fine-Tune & Export (free Colab T4)

Fine-tunes **`unsloth/Qwen3.5-0.8B`** on ShadowTalk's SFT dataset (`shadowtalk_sft.jsonl`, 620
examples: 45% shadowvisual JSON / 25% agentic missions / 20% grounded RAG / 10% identity+multilingual)
and exports **three artifacts**:

| Artifact | Repo | Consumer |
|---|---|---|
| Merged 16-bit weights | `shadowtalkai/Shadow-1-Whisper-0.8B` | base for GGUF + ONNX transplant |
| GGUF `q4_k_m` | `shadowtalkai/Shadow-1-Whisper-0.8B-GGUF` | Tauri Desktop (llama.cpp) |
| ONNX q4 (WebGPU) | `shadowtalkai/Shadow-1-Whisper-0.8B-ONNX` | Browser worker (`@huggingface/transformers` v4+) |

**Before running:** get a Hugging Face **write** token (fine-grained, repos: *write*), create the three
repos under `shadowtalkai/` (or let Cell 4 create them), and upload `shadowtalk_sft.jsonl` next to this
notebook. Everything below runs on a free T4 (~3 GB VRAM for LoRA, fp16).

## Cell 1 — Setup
Fails fast if the runtime has no GPU, then installs the stack **exactly as the official
unsloth Qwen3.5 Vision Colab notebook does** — pinned `torch 2.8.0`, `transformers==5.2.0`,
`trl==0.22.2`, `tokenizers≤0.23.0`, xformers + `unsloth install-kernels` — plus the
export-side ONNX tools. Takes ~3-5 min on a fresh Colab runtime.

In [ ]:
# 1 ─ Setup — fail-fast GPU check, then the PINNED install stack
#
# Check the GPU BEFORE installing anything: on a CPU runtime this dies in 2s
# instead of after 3-5 min of pip. torch ships preinstalled on Colab.
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}  |  capability {torch.cuda.get_device_capability(0)}")

# Install the EXACT combo the official unsloth Qwen3.5 Vision Colab notebook
# (unslothai/notebooks · nb/Qwen3_5_(0_8B)_Vision.ipynb — the reference notebook
# for our base model) installs. The pins are load-bearing: PyPI's latest
# transformers is 5.24.x, but unsloth's Qwen3.5 kernels are built and tested
# against transformers==5.2.0 + trl==0.22.2 + tokenizers<=0.23.0 — a loose
# 'transformers>=5.2.0' resolves to an untested pairing and breaks the patched
# forward pass / SFTConfig API.
!pip install --upgrade -qqq uv
import importlib.util, os
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

# Export-side tools for Cell 4/5 (ONNX graph verify + tokenizer export). Deliberately
# WITHOUT -U so nothing drags huggingface_hub/safetensors away from what
# transformers==5.2.0 pins.
!pip install -q onnx onnxruntime sentencepiece protobuf

# Verify the pinned combo before burning GPU time. If an assert ever fires
# mid-session: Runtime → Restart session, then re-run this cell.
import importlib.metadata as im
for pkg in ["unsloth", "unsloth_zoo", "transformers", "tokenizers", "trl", "peft",
            "bitsandbytes", "onnx", "onnxruntime", "huggingface_hub"]:
    try:
        print(f"{pkg:18s} {im.version(pkg)}")
    except im.PackageNotFoundError:
        print(f"{pkg:18s} MISSING")
assert im.version("transformers") == "5.2.0", "transformers must be exactly 5.2.0 (unsloth Qwen3.5 tested pin)"
assert im.version("trl") == "0.22.2", "trl must be exactly 0.22.2 (official Qwen3.5 Vision notebook pin)"

# T4 note: Turing (sm_75) has no native bf16 — 'False' here is EXPECTED, not an
# error. Cell 3 auto-selects fp16 on T4 (bf16 only on Ampere+/L4).
print("bf16 supported:", torch.cuda.is_bf16_supported())


## Cell 2 — Auth & Model Load
Prompts for the HF token securely (`getpass`, never printed), loads the base model in **bf16/fp16 at full
precision** (no 4-bit: a 0.8B model + LoRA fits comfortably in ~3 GB on the T4), and attaches LoRA adapters
to the **language layers only** — the vision stack is frozen because Shadow-1-Whisper is a text-contract model.

In [ ]:
# 2 ─ Auth & model load (secure token + LoRA on language layers)
import getpass
from huggingface_hub import login

HF_TOKEN = getpass.getpass("🔑 Paste your Hugging Face WRITE token (hf_...): ")
assert HF_TOKEN.startswith("hf_"), "That doesn't look like an HF token (should start with 'hf_')"
login(token=HF_TOKEN)

from unsloth import FastVisionModel

BASE_MODEL = "unsloth/Qwen3.5-0.8B"
SEED = 3407

# load_in_4bit=False → full-precision LoRA: ~3 GB VRAM on the T4 (fp16; bf16 on Ampere+)
model, processor = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth",
)

# Attach LoRA — language layers only, vision frozen (text-contract model)
model = FastVisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
)

FastVisionModel.for_training(model)
model.print_trainable_parameters()

tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor
print("Base model loaded:", BASE_MODEL)
print("dtype:", next(model.parameters()).dtype)

## Cell 3 — Train (SFT on the chat-template)
Loads `shadowtalk_sft.jsonl` (upload it next to the notebook — the cell prompts if missing), renders every
record through `tokenizer.apply_chat_template(..., add_generation_prompt=False)` and trains with
`SFTTrainer` for **3 epochs @ lr 2e-4**, seq len 2048. On a free T4 expect **~25-40 min** for the 620-example run.

In [ ]:
# 3 ─ Train — chat-templated SFT

# Loss-surface note: SFT loss covers all assistant turns, so tool-card turns inside
# multi-turn records must never be reachable as model output. Every card-bearing record
# in shadowtalk_sft.jsonl opens with a request the runtime tool detector
# (useToolOrchestrator.detectToolSync) intercepts before the model is called — calc/
# calculate prefix, bare arithmetic, 'search the web for …', or an explicit /image
# command. The only exceptions are database_query / file_manager, which have no
# auto-detection: their records intentionally teach the honest 'not yet deployed'
# runtime reply instead of a fabricated tool success.
import json, os
from pathlib import Path
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

DATA_FILE = "shadowtalk_sft.jsonl"
if not Path(DATA_FILE).exists():
    print("Upload shadowtalk_sft.jsonl now (from training/shadow-1-whisper/)…")
    from google.colab import files
    uploaded = files.upload()
    DATA_FILE = list(uploaded.keys())[0]

rows = [json.loads(line) for line in open(DATA_FILE, encoding="utf-8")]
assert rows and all("messages" in r for r in rows), "dataset rows must be {'messages': [...]}"
print(f"loaded {len(rows)} examples")

# Render each record with the model's own chat template.
# add_generation_prompt=False → the assistant turn is TRAINING TARGET, not a prompt prefix.
def render(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list([render(r) for r in rows]).shuffle(seed=3407)
print("sample:\n" + dataset[0]["text"][:600] + "\n…")

MAX_SEQ = 2048
cfg = SFTConfig(
    output_dir="shadow-1-ckpt",
    dataset_text_field="text",
    max_length=MAX_SEQ,                     # trl 0.22.2 renamed max_seq_length → max_length
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,          # effective batch 16
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    fp16=(not torch.cuda.is_bf16_supported()),  # T4 = fp16; A100/L4 = bf16
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    logging_steps=5,
    save_strategy="epoch",
    seed=3407,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=cfg,
)

trainer_stats = trainer.train()

gpu_s = round(trainer_stats.metrics.get("train_runtime", 0), 1)
print(f"trained in {gpu_s}s  |  peak mem ≈ {torch.cuda.max_memory_reserved()/1e9:.2f} GB")

## Cell 4 — Push Merged 16-bit + GGUF q4_k_m
Merges the LoRA adapters back into the base weights, pushes the merged model to
`shadowtalkai/Shadow-1-Whisper-0.8B`, then quantizes to **GGUF `q4_k_m`** and pushes
`shadowtalkai/Shadow-1-Whisper-0.8B-GGUF` (desktop/Tauri runtime). GGUF quantization runs ~8-15 min on the T4.
Both repos are created **private** — flip them public after you've eyeballed them (see README).

In [ ]:
# 4 ─ Push merged 16-bit + GGUF q4_k_m
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)

MERGED_REPO = "shadowtalkai/Shadow-1-Whisper-0.8B"
GGUF_REPO = "shadowtalkai/Shadow-1-Whisper-0.8B-GGUF"

for repo in (MERGED_REPO, GGUF_REPO):
    api.create_repo(repo_id=repo, repo_type="model", private=True, exist_ok=True)

# 4a — merged 16-bit (the canonical weights + tokenizer + chat template)
model.push_to_hub_merged(
    MERGED_REPO,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN,
)
print(f"✅ merged 16-bit → https://huggingface.co/{MERGED_REPO}")

# 4b — GGUF q4_k_m for the Tauri desktop runtime (llama.cpp)
model.push_to_hub_gguf(
    GGUF_REPO,
    tokenizer,
    quantization_method="q4_k_m",
    token=HF_TOKEN,
)
print(f"✅ GGUF q4_k_m → https://huggingface.co/{GGUF_REPO}")

## Cell 5 — Export the WebGPU ONNX package
`Qwen3.5-0.8B` is a **hybrid Gated DeltaNet + vision** architecture — we do NOT re-export the graph from
torch (fragile, slow, and would lose the battle-tested WebGPU kernels). Instead we **transplant**:

1. download the reference WebGPU ONNX graph from `onnx-community/Qwen3.5-0.8B-ONNX`,
2. overwrite every language-side initializer (attention, MLP, embeddings, Gated DeltaNet projections)
   with the fine-tuned merged weights — matched by normalized tensor names,
3. swap in the fine-tuned tokenizer + `chat_template` + `generation_config`,
4. verify tensor shapes with **onnxruntime** (every rewritten graph must load and produce the documented
   input/output signature), and
5. upload to `shadowtalkai/Shadow-1-Whisper-0.8B-ONNX`.

The transplant reports matched/unmatched initializer counts and **fails loudly** below a 70% coverage
threshold — a silently half-transplanted model is worse than a failed export.

In [ ]:
# 5 ─ ONNX transplant: fine-tuned language weights → reference WebGPU graph
import glob, json, shutil
from pathlib import Path

import numpy as np
import onnx
from onnx import TensorProto, numpy_helper
from huggingface_hub import snapshot_download

REF_REPO = "onnx-community/Qwen3.5-0.8B-ONNX"
MERGED_DIR = "Shadow-1-Whisper-0.8B-merged"   # local copy of cell-4a output
ONNX_REPO = "shadowtalkai/Shadow-1-Whisper-0.8B-ONNX"

if not Path(MERGED_DIR).exists():
    # cell 4 pushed to the Hub; pull the same merged weights back down
    from huggingface_hub import snapshot_download as _sd
    MERGED_DIR = str(_sd(MERGED_REPO, local_dir="Shadow-1-Whisper-0.8B-merged"))

# 5.1 — reference graph → working dir
work = Path("Shadow-1-Whisper-0.8B-ONNX-work")
shutil.rmtree(work, ignore_errors=True)
ref_dir = snapshot_download(REF_REPO)
shutil.copytree(ref_dir, work, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".cache"))
print("reference graph files:", len(list(work.rglob('*.onnx'))))

# 5.2 — merged state dict (safetensors shards)
from safetensors.torch import load_file
state = {}
for shard in sorted(glob.glob(f"{MERGED_DIR}/*.safetensors")):
    state.update(load_file(shard))
print(f"merged state dict: {len(state)} tensors")

def find_weight(init_name: str):
    """Match an ONNX initializer name to the HF state dict, tolerating the
    naming drift between graph exporters and transformers (prefixes, onnx:: glue)."""
    candidates = [
        init_name,
        init_name.removeprefix("module."),
        init_name.split("onnx::")[-1],
        init_name.replace("onnx::MatMul_", "").replace("onnx::MatMulInverse_", ""),
        init_name.replace("onnx::", "").replace("~", "."),
    ]
    for cand in candidates:
        if cand in state:
            return cand
    # suffix match — graph prefixes like "model.encoder." vs plain HF names
    for sk in state:
        if init_name.endswith(sk) or sk.endswith(init_name):
            return sk
    return None

MIN_COVERAGE = 0.70
report = {}
for onnx_path in sorted(work.rglob("*.onnx")):
    m = onnx.load(str(onnx_path), load_external_data=True)
    matched, mismatched, missing = 0, [], []
    total = len(m.graph.initializer)
    for init in m.graph.initializer:
        sk = find_weight(init.name)
        if sk is None:
            missing.append(init.name)
            continue
        arr = state[sk].detach().cpu().numpy()
        dims = list(init.dims)
        if dims and list(arr.shape) != dims:
            if arr.size == int(np.prod(dims)):
                arr = arr.reshape(dims)
            else:
                mismatched.append(init.name)
                continue
        # keep the graph's declared dtype (fp16 graphs stay fp16)
        if init.data_type == TensorProto.FLOAT16:
            arr = arr.astype(np.float16)
        elif init.data_type == TensorProto.FLOAT:
            arr = arr.astype(np.float32)
        new = numpy_helper.from_array(arr, name=init.name)
        init.CopyFrom(new)
        matched += 1
    coverage = matched / max(1, total)
    report[str(onnx_path.relative_to(work))] = {
        "initializers": total, "matched": matched,
        "missing": len(missing), "shape_mismatch": len(mismatched),
        "coverage": round(coverage, 3),
    }
    onnx.save_model(
        m, str(onnx_path),
        save_as_external_data=True,
        all_tensors_to_one_file=True,
        location=onnx_path.name + ".data",   # relative to the model file's own dir
        size_threshold=1024,
    )

print(json.dumps(report, indent=2))
best = max(r["coverage"] for r in report.values())
assert best >= MIN_COVERAGE, (
    f"transplant coverage {best:.0%} < {MIN_COVERAGE:.0%} — graph naming drift too large; "
    "inspect report and extend find_weight() before shipping."
)

# 5.3 — tokenizer + configs from the merged model (chat_template included)
TOKEN_FILES = ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
               "generation_config.json", "chat_template.jinja"]
for name in TOKEN_FILES:
    src = Path(MERGED_DIR) / name
    if src.exists():
        shutil.copy(src, work / name)
# older transformers keep the template inside tokenizer_config.json
tc_path = work / "tokenizer_config.json"
if tc_path.exists() and not (work / "chat_template.jinja").exists():
    cfg_json = json.loads(tc_path.read_text())
    if "chat_template" in cfg_json:
        (work / "chat_template.jinja").write_text(cfg_json["chat_template"])
print("tokenizer + configs staged")

# 5.4 — onnxruntime shape verification (graph must LOAD and expose the documented IO)
import onnxruntime as ort
print("ORT providers:", ort.get_available_providers())
main_models = sorted(work.rglob("*.onnx"))
for p in main_models:
    sess = ort.InferenceSession(str(p), providers=["CPUExecutionProvider"])
    ins = [(i.name, i.shape) for i in sess.get_inputs()]
    outs = [(o.name, o.shape) for o in sess.get_outputs()]
    print(f"✓ {p.relative_to(work)}  inputs={ins[:3]}… outputs={outs[:2]}…")

# 5.5 — upload (private; flip public after inspection)
api.create_repo(ONNX_REPO, repo_type="model", private=True, exist_ok=True)
api.upload_folder(folder_path=str(work), repo_id=ONNX_REPO, repo_type="model", commit_message="Shadow-1-Whisper: transplanted language weights + tokenizer onto WebGPU ONNX graph")
print(f"✅ ONNX package → https://huggingface.co/{ONNX_REPO}")

print("""
NEXT — flip the three repos PUBLIC, then wire the web runtime:
  Qwen3_5ForConditionalGeneration + AutoProcessor from
  "shadowtalkai/Shadow-1-Whisper-0.8B-ONNX", dtype q4, device webgpu,
  temperature 0.7 / top_p 0.8 / top_k 20 / presence_penalty 1.5
(see README.md — WebGPU worker wiring section)
""")